# تدريب نموذج YOLOv8 لكشف لوحات السيارات

**الهدف:** تدريب نموذج خاص بك من بياناتك الموسّمة (دمج 4 datasets للوحات السعودية)، تحصل في النهاية على ملف `best.pt` مستقل تستخدمه إلى الأبد بدون أي مصادقة.

**Dataset المُجهّز:**
- 2516 صورة تدريب
- 365 صورة validation
- 41 صورة اختبار
- فئة واحدة: `license_plate`

**سير العمل:**
1. التحقق من GPU
2. تثبيت Ultralytics
3. رفع 4 أجزاء الـ dataset ودمجها
4. استخراج الـ dataset والتحقق من بنيته
5. عينة بصرية للتحقق
6. تدريب YOLOv8
7. تقييم النموذج
8. اختبار على صور جديدة
9. تنزيل `best.pt`

**قبل البدء:** فعّل GPU من `Runtime → Change runtime type → T4 GPU`

## 1. التحقق من GPU

In [ ]:
!nvidia-smi

## 2. تثبيت المكتبات

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()

## 3. رفع أجزاء الـ Dataset ودمجها

بسبب حجم الـ dataset (90 MB)، تم تقسيمه إلى 4 أجزاء. شغّل الخلية التالية واختر **جميع الـ 4 ملفات** (Ctrl+click في نافذة الاختيار):
- `merged_part_aa`
- `merged_part_ab`
- `merged_part_ac`
- `merged_part_ad`

سيتم دمجها تلقائياً في Colab.

In [ ]:
import os
import shutil
from google.colab import files

# تنظيف
WORK_DIR = "/content"
for f in os.listdir(WORK_DIR):
    if f.startswith("merged_part_") or f == "merged_license_plate_dataset.zip":
        os.remove(os.path.join(WORK_DIR, f))

print("اختر الـ 4 أجزاء (merged_part_aa, ab, ac, ad)...")
print("💡 يمكنك تحديدهم كلهم دفعة واحدة بـ Ctrl+click\n")
uploaded = files.upload()

# التحقق من رفع كل الأجزاء
parts = sorted([f for f in uploaded.keys() if f.startswith("merged_part_")])
print(f"\nتم رفع {len(parts)} أجزاء:")
for p in parts:
    size_mb = os.path.getsize(p) / 1e6
    print(f"  ✓ {p} ({size_mb:.1f} MB)")

assert len(parts) == 4, f"⚠️ يجب أن يكون عدد الأجزاء 4، تم رفع {len(parts)} فقط"

# دمج الأجزاء
ZIP_PATH = "/content/merged_license_plate_dataset.zip"
print("\nجاري الدمج...")
with open(ZIP_PATH, "wb") as out:
    for p in parts:
        with open(p, "rb") as inp:
            out.write(inp.read())
        os.remove(p)  # حذف الجزء بعد الدمج لتوفير المساحة

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f"✅ تم الدمج: {ZIP_PATH} ({size_mb:.1f} MB)")

## 4. استخراج الـ Dataset

In [ ]:
import zipfile
import yaml
from pathlib import Path

DATASET_PARENT = "/content/dataset"
if os.path.exists(DATASET_PARENT):
    shutil.rmtree(DATASET_PARENT)
os.makedirs(DATASET_PARENT, exist_ok=True)

# فك الضغط
print("جاري فك الضغط...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(DATASET_PARENT)

# البحث عن data.yaml
yaml_files = list(Path(DATASET_PARENT).rglob("data.yaml"))
assert len(yaml_files) > 0, "⚠️ data.yaml غير موجود"
DATA_YAML = str(yaml_files[0])
dataset_root = Path(DATA_YAML).parent
print(f"جذر الـ dataset: {dataset_root}")

# تصحيح المسار في data.yaml ليتطابق مع موقعه في Colab
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = str(dataset_root.resolve())
cfg['train'] = 'train/images'
cfg['val'] = 'valid/images'
if (dataset_root / 'test' / 'images').exists():
    cfg['test'] = 'test/images'
with open(DATA_YAML, 'w') as f:
    yaml.dump(cfg, f, sort_keys=False)

print(f"\nمحتوى data.yaml:")
with open(DATA_YAML) as f:
    print(f.read())

# عدّ الصور
for split in ['train', 'valid', 'test']:
    img_dir = dataset_root / split / 'images'
    if img_dir.exists():
        n = len(list(img_dir.glob('*')))
        print(f"  {split}: {n} صورة")

## 5. عرض عينة من البيانات للتحقق البصري

In [ ]:
import matplotlib.pyplot as plt
import cv2
import random

random.seed(42)
train_images = list((dataset_root / 'train' / 'images').glob('*'))
labels_dir = dataset_root / 'train' / 'labels'

# اختر صورتين من كل مصدر للتحقق من تنوع البيانات
samples = []
for prefix in ['ds1', 'ds2', 'ds3', 'ds4']:
    matching = [im for im in train_images if im.name.startswith(prefix + '_')]
    samples.extend(random.sample(matching, min(2, len(matching))))

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for ax, img_path in zip(axes, samples):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    label_path = labels_dir / (img_path.stem + '.txt')
    if label_path.exists():
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls, xc, yc, bw, bh = map(float, parts[:5])
                    x1 = int((xc - bw/2) * w)
                    y1 = int((yc - bh/2) * h)
                    x2 = int((xc + bw/2) * w)
                    y2 = int((yc + bh/2) * h)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 4)
    
    src = img_path.name.split('_')[0]
    ax.imshow(img)
    ax.set_title(f"{src}", fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()
print("\nيجب أن ترى مربعات خضراء حول اللوحات في كل صورة.")

## 6. تدريب النموذج

**التوصية مع ~2900 صورة:** نبدأ بـ `yolov8s.pt` (small) — توازن ممتاز بين السرعة والدقة.

**الوقت المتوقع على T4:** ~40-50 دقيقة لـ 50 epoch.

**ملاحظات:**
- `patience=20` يوقف التدريب تلقائياً لو لم يتحسن النموذج لـ 20 epoch متتالية
- `mosaic=1.0` يساعد كثيراً مع البيانات المتنوعة
- `flipud=0` لأن اللوحات لا تكون مقلوبة عمودياً في الواقع

In [ ]:
from ultralytics import YOLO

BASE_MODEL = "yolov8s.pt"   # n=أسرع، s=متوازن، m=أدق
EPOCHS = 50
IMG_SIZE = 640
BATCH_SIZE = 16            # قلّل لـ 8 لو نفدت ذاكرة GPU
PROJECT = "/content/runs"
RUN_NAME = "license_plate_v1"

model = YOLO(BASE_MODEL)

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    patience=20,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    verbose=True,
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.0,
    fliplr=0.5,
    save=True,
    plots=True
)

print(f"\n✅ التدريب اكتمل!")
print(f"النموذج: {PROJECT}/{RUN_NAME}/weights/best.pt")

## 7. عرض رسوم نتائج التدريب

In [ ]:
from IPython.display import Image, display

results_dir = f"{PROJECT}/{RUN_NAME}"
for plot_file in ['results.png', 'confusion_matrix.png', 'PR_curve.png', 'F1_curve.png']:
    plot_path = os.path.join(results_dir, plot_file)
    if os.path.exists(plot_path):
        print(f"\n📊 {plot_file}")
        display(Image(plot_path))

## 8. تقييم النموذج على Validation Set

In [ ]:
best_model_path = f"{results_dir}/weights/best.pt"
best_model = YOLO(best_model_path)
metrics = best_model.val(data=DATA_YAML)

print("\n" + "="*60)
print("معايير الأداء على Validation Set")
print("="*60)
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")

print("\n📌 تفسير الأرقام:")
print("  • mAP@0.5 ≥ 0.90 → ممتاز، النموذج جاهز للإنتاج")
print("  • mAP@0.5 0.75-0.90 → جيد، يمكن تحسينه بـ epochs إضافية")
print("  • mAP@0.5 < 0.75 → يحتاج بيانات أكثر أو نموذج أكبر (m/l)")

## 9. اختبار على Test Set

In [ ]:
import math

test_dir = dataset_root / 'test' / 'images'
if test_dir.exists():
    test_images = sorted(list(test_dir.glob('*')))[:9]
    n = len(test_images)
    
    if n > 0:
        cols = 3
        rows = math.ceil(n / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(16, 5*rows))
        axes = axes.flatten() if n > 1 else [axes]
        
        for i, img_path in enumerate(test_images):
            results = best_model.predict(str(img_path), conf=0.25, verbose=False)
            annotated = results[0].plot()
            annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
            axes[i].imshow(annotated_rgb)
            axes[i].set_title(f"{img_path.name[:30]} | {len(results[0].boxes)} لوحة", fontsize=9)
            axes[i].axis('off')
        
        for j in range(n, len(axes)):
            axes[j].axis('off')
        plt.tight_layout()
        plt.show()

## 10. اختبار على صور جديدة من جهازك (اختياري)

In [ ]:
TEST_DIR = "/content/my_test_images"
if os.path.exists(TEST_DIR):
    shutil.rmtree(TEST_DIR)
os.makedirs(TEST_DIR, exist_ok=True)

print("ارفع صوراً جديدة لاختبار النموذج (يمكنك تخطّي هذه الخلية)...")
uploaded = files.upload()

for fname in uploaded.keys():
    shutil.move(fname, os.path.join(TEST_DIR, fname))

my_test_images = sorted([os.path.join(TEST_DIR, f) for f in os.listdir(TEST_DIR)
                          if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))])

if my_test_images:
    n = min(len(my_test_images), 9)
    cols = 3
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5*rows))
    axes = axes.flatten() if n > 1 else [axes]
    
    for i, img_path in enumerate(my_test_images[:n]):
        results = best_model.predict(img_path, conf=0.25, verbose=False)
        annotated = results[0].plot()
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        axes[i].imshow(annotated_rgb)
        axes[i].set_title(f"{os.path.basename(img_path)[:30]} | {len(results[0].boxes)} لوحة", fontsize=9)
        axes[i].axis('off')
    
    for j in range(n, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()

## 11. تنزيل النموذج المُدرَّب

In [ ]:
FINAL_MODEL = "/content/license_plate_detector.pt"
shutil.copy(best_model_path, FINAL_MODEL)

print(f"حجم الملف: {os.path.getsize(FINAL_MODEL)/1e6:.1f} MB")
print("جاري التنزيل (1/2): النموذج...")
files.download(FINAL_MODEL)

# تنزيل كل نتائج التدريب (curves, metrics, weights, config)
shutil.make_archive("/content/training_results", 'zip', results_dir)
print("\nجاري التنزيل (2/2): نتائج التدريب...")
files.download("/content/training_results.zip")

print("\n✅ تم! ستجد في جهازك:")
print("  1. license_plate_detector.pt - النموذج جاهز للاستخدام")
print("  2. training_results.zip - تفاصيل التدريب (curves, metrics)")

## 12. استخدام النموذج محلياً

بعد تنزيل `license_plate_detector.pt`، في جهازك:

```python
from ultralytics import YOLO

# مستقل بالكامل - لا يحتاج إنترنت ولا مصادقة
model = YOLO('license_plate_detector.pt')

results = model.predict('car.jpg', conf=0.25)

for box in results[0].boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    conf = float(box.conf[0])
    print(f'لوحة عند ({x1:.0f}, {y1:.0f}) - ({x2:.0f}, {y2:.0f}) | ثقة: {conf:.2f}')
    
    # قص اللوحة وتمريرها لنموذج الـ OCR الخاص بك
    import cv2
    img = cv2.imread('car.jpg')
    plate_crop = img[int(y1):int(y2), int(x1):int(x2)]
    cv2.imwrite('plate.jpg', plate_crop)
```